# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
spaceship.shape

(8693, 14)

**Check for data types**

In [4]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [5]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
df = spaceship.dropna()
print(f"Filas originales: {len(spaceship)}")
print(f"Filas después de eliminar: {len(df)}")
print(f"Filas eliminadas: {len(spaceship) - len(df)}")

Filas originales: 8693
Filas después de eliminar: 6606
Filas eliminadas: 2087


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
df['Cabin'] = df['Cabin'].str[0]

df['Cabin'].value_counts()

C:\Users\gcast\AppData\Local\Temp\ipykernel_29676\2362934069.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cabin'] = df['Cabin'].str[0]


Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

- Drop PassengerId and Name

In [8]:
df = df.drop(columns=['Name', 'PassengerId'])

- For non-numerical columns, do dummies.

In [ ]:
df['CryoSleep'] = df['CryoSleep'].astype(int)
df['VIP'] = df['VIP'].astype(int)
df['Transported'] = df['Transported'].astype(int)

df = pd.get_dummies(df, columns=['HomePlanet', 'Destination', 'Cabin'], drop_first=True)

columns_dummed =[
    'HomePlanet_Europa',
    'HomePlanet_Mars',
    'Destination_PSO J318.5-22',
    'Destination_TRAPPIST-1e',
    'Cabin_B',
    'Cabin_C',
    'Cabin_D',
    'Cabin_E',
    'Cabin_F',
    'Cabin_G',
    'Cabin_T'
    ]
df[columns_dummed] = df[columns_dummed].astype(int)

**Perform Train Test Split**

In [32]:
y = df['Transported']
X = df.drop(columns=['Transported'])

print(X.info())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")

<class 'pandas.core.frame.DataFrame'>
Index: 6606 entries, 0 to 8692
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CryoSleep                  6606 non-null   int64  
 1   Age                        6606 non-null   float64
 2   VIP                        6606 non-null   int64  
 3   RoomService                6606 non-null   float64
 4   FoodCourt                  6606 non-null   float64
 5   ShoppingMall               6606 non-null   float64
 6   Spa                        6606 non-null   float64
 7   VRDeck                     6606 non-null   float64
 8   HomePlanet_Europa          6606 non-null   int64  
 9   HomePlanet_Mars            6606 non-null   int64  
 10  Destination_PSO J318.5-22  6606 non-null   int64  
 11  Destination_TRAPPIST-1e    6606 non-null   int64  
 12  Cabin_B                    6606 non-null   int64  
 13  Cabin_C                    6606 non-null   int64  
 1

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [33]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

predictions = knn.predict(X_test)

- Evaluate your model's performance. Comment it

In [35]:
from sklearn.metrics import accuracy_score, classification_report

print(f"Model precision: {accuracy_score(y_test, predictions)}")
print(classification_report(y_test, predictions))

Model precision: 0.7904689863842662
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       653
           1       0.80      0.79      0.79       669

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

